# **Relational Chess Tournament & Player Performance Dashboard**

Tables
------
- **tournaments**     (tournament_id, tournament_name UNIQUE, arbiter_email)
- **player_entries**  (entry_id, tournament_id -> tournaments  ON DELETE CASCADE, player_name, entry_date)
- **game_records**    (record_id, entry_id -> player_entries   ON DELETE CASCADE, record_type, record_value)

In [1]:
import html
import math
import re
import sqlite3
from datetime import date

DB_PATH = "chess_federation.db"

# 1. DATABASE LAYER

In [2]:
def get_conn():
    """New connection with foreign keys ON (SQLite disables them by default,
    and the setting is per-connection, so it must be set every time)."""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


def init_db():
    with get_conn() as conn:
        conn.executescript(
            """
            CREATE TABLE IF NOT EXISTS tournaments (
                tournament_id   INTEGER PRIMARY KEY AUTOINCREMENT,
                tournament_name TEXT NOT NULL UNIQUE COLLATE NOCASE,
                arbiter_email   TEXT NOT NULL
            );

            CREATE TABLE IF NOT EXISTS player_entries (
                entry_id      INTEGER PRIMARY KEY AUTOINCREMENT,
                tournament_id INTEGER NOT NULL,
                player_name   TEXT NOT NULL,
                entry_date    TEXT NOT NULL,
                FOREIGN KEY (tournament_id) REFERENCES tournaments(tournament_id)
                    ON DELETE CASCADE
            );

            CREATE TABLE IF NOT EXISTS game_records (
                record_id    INTEGER PRIMARY KEY AUTOINCREMENT,
                entry_id     INTEGER NOT NULL,
                record_type  TEXT NOT NULL,
                record_value REAL NOT NULL CHECK (record_value >= 0),
                FOREIGN KEY (entry_id) REFERENCES player_entries(entry_id)
                    ON DELETE CASCADE
            );
            """
        )

def seed_if_empty():
    """Seed sample data only if all three tables are completely empty."""
    with get_conn() as conn:
        total = sum(
            conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
            for t in ("tournaments", "player_entries", "game_records")
        )
        if total:
            return
        tournaments = [
            ("Tata Steel Masters 2026", "arbiter.tata@chessfed.org"),
            ("Candidates Tournament 2026", "arbiter.candidates@chessfed.org"),
            ("Chennai Grand Masters Open", "arbiter.chennai@chessfed.org"),
        ]
        conn.executemany(
            "INSERT INTO tournaments (tournament_name, arbiter_email) VALUES (?, ?)",
            tournaments,
        )
        ids = {n: i for i, n in conn.execute(
            "SELECT tournament_id, tournament_name FROM tournaments")}
        entries = [
            ("Tata Steel Masters 2026", "R. Praggnanandhaa", "2026-01-10"),
            ("Tata Steel Masters 2026", "D. Gukesh", "2026-01-10"),
            ("Candidates Tournament 2026", "D. Gukesh", "2026-03-02"),
            ("Candidates Tournament 2026", "A. Erigaisi", "2026-03-03"),
            ("Chennai Grand Masters Open", "R. Praggnanandhaa", "2026-06-15"),
        ]
        entry_ids = []
        for t, p, d in entries:
            cur = conn.execute(
                "INSERT INTO player_entries (tournament_id, player_name, entry_date) "
                "VALUES (?, ?, ?)", (ids[t], p, d))
            entry_ids.append(cur.lastrowid)
        records = [
            (entry_ids[0], "Opponent Rating", 2750),
            (entry_ids[0], "Points Scored", 1),
            (entry_ids[1], "Moves Played", 42),
            (entry_ids[2], "Opponent Rating", 2801),
            (entry_ids[3], "Points Scored", 0.5),
            (entry_ids[4], "Moves Played", 57),
        ]
        conn.executemany(
            "INSERT INTO game_records (entry_id, record_type, record_value) "
            "VALUES (?, ?, ?)", records)

# Validation

In [3]:
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")


def parse_record_value(text):
    """Return a non-negative finite float, or raise ValueError with a message."""
    text = (text or "").strip()
    try:
        v = float(text)
    except ValueError:
        raise ValueError("Record Value must be a number.")
    if math.isnan(v) or math.isinf(v):
        raise ValueError("Record Value must be a finite number.")
    if v < 0:
        raise ValueError("Record Value cannot be negative.")
    return v


def add_record(tournament_name, player_name, record_type, record_value_text,
               arbiter_email=""):
    """One submission = (find or create tournament) + 1 player entry + 1 game
    record, all in a single transaction. Returns a status message."""
    tournament_name = (tournament_name or "").strip()
    player_name = (player_name or "").strip()
    record_type = (record_type or "").strip()
    arbiter_email = (arbiter_email or "").strip()

    if not tournament_name or not player_name or not record_type \
            or not (record_value_text or "").strip():
        raise ValueError("Tournament Name, Player Name, Record Type and "
                         "Record Value must all be filled in.")
    value = parse_record_value(record_value_text)

    conn = get_conn()
    try:
        row = conn.execute(
            "SELECT tournament_id FROM tournaments WHERE tournament_name = ?",
            (tournament_name,)).fetchone()
        if row:
            tid, created = row[0], False
        else:
            if not arbiter_email:
                raise ValueError("This is a new tournament - please enter an "
                                 "Arbiter Email for it.")
            if not EMAIL_RE.match(arbiter_email):
                raise ValueError("Arbiter Email doesn't look like a valid address.")
            tid = conn.execute(
                "INSERT INTO tournaments (tournament_name, arbiter_email) VALUES (?, ?)",
                (tournament_name, arbiter_email)).lastrowid
            created = True
        eid = conn.execute(
            "INSERT INTO player_entries (tournament_id, player_name, entry_date) "
            "VALUES (?, ?, ?)", (tid, player_name, date.today().isoformat())).lastrowid
        conn.execute(
            "INSERT INTO game_records (entry_id, record_type, record_value) "
            "VALUES (?, ?, ?)", (eid, record_type, value))
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

    prefix = "Created new tournament and added" if created else "Added"
    return f"{prefix} entry #{eid} for {player_name} in '{tournament_name}'."

# Delete helpers

In [4]:
DELETE_KINDS = {
    "Chess Tournament": ("tournaments", "tournament_id"),
    "Player Entry": ("player_entries", "entry_id"),
    "Game Record": ("game_records", "record_id"),
}


def get_delete_options(kind):
    """List of (label, id) for the chosen kind, always built from live data."""
    with get_conn() as conn:
        if kind == "Chess Tournament":
            rows = conn.execute(
                "SELECT tournament_id, tournament_name FROM tournaments "
                "ORDER BY tournament_id").fetchall()
            return [(f"#{i} - {n}", i) for i, n in rows]
        if kind == "Player Entry":
            rows = conn.execute(
                "SELECT e.entry_id, e.player_name, t.tournament_name "
                "FROM player_entries e JOIN tournaments t USING (tournament_id) "
                "ORDER BY e.entry_id").fetchall()
            return [(f"Entry #{i} - {p} ({t})", i) for i, p, t in rows]
        if kind == "Game Record":
            rows = conn.execute(
                "SELECT g.record_id, g.record_type, g.record_value, e.player_name "
                "FROM game_records g JOIN player_entries e USING (entry_id) "
                "ORDER BY g.record_id").fetchall()
            return [(f"Record #{i} - {p}: {rt} = {fmt(v)}", i)
                    for i, rt, v, p in rows]
    return []


def delete_row(kind, row_id):
    table, pk = DELETE_KINDS[kind]
    with get_conn() as conn:  # foreign_keys=ON -> cascades fire
        cur = conn.execute(f"DELETE FROM {table} WHERE {pk} = ?", (row_id,))
        if cur.rowcount == 0:
            raise ValueError("That row no longer exists.")
    return f"Deleted {kind.lower()} #{row_id} (and any dependent child rows)."


def fmt(v):
    """Show 2750.0 as 2750 but keep 0.5 as 0.5."""
    return f"{v:g}" if isinstance(v, float) else str(v)

# Table views

In [5]:
VIEW_QUERIES = {
    # LEFT JOIN so an entry whose last game record was deleted stays visible.
    "Combined View": """
        SELECT e.entry_id AS "Entry ID", t.tournament_name AS "Tournament Name",
               e.player_name AS "Player Name", e.entry_date AS "Entry Date",
               g.record_type AS "Record Type", g.record_value AS "Record Value"
        FROM player_entries e
        JOIN tournaments t ON t.tournament_id = e.tournament_id
        LEFT JOIN game_records g ON g.entry_id = e.entry_id
        ORDER BY e.entry_id, g.record_id""",
    "tournaments": "SELECT * FROM tournaments ORDER BY tournament_id",
    "player_entries": "SELECT * FROM player_entries ORDER BY entry_id",
    "game_records": "SELECT * FROM game_records ORDER BY record_id",
}


def query_table_html(sql):
    """Run a query and return a self-contained HTML table (no pandas needed)."""
    with get_conn() as conn:
        cur = conn.execute(sql)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    th = "".join(
        f"<th style='text-align:left;padding:6px 12px;background:#2b3a55;"
        f"color:#fff'>{html.escape(c)}</th>" for c in cols)
    if not rows:
        body = (f"<tr><td colspan='{len(cols)}' style='padding:10px;"
                f"color:#888'>No rows.</td></tr>")
    else:
        body = ""
        for i, r in enumerate(rows):
            bg = "#f3f5f9" if i % 2 else "#ffffff"
            tds = "".join(
                f"<td style='padding:5px 12px;background:{bg};color:#111'>"
                f"{html.escape('' if v is None else fmt(v))}</td>" for v in r)
            body += f"<tr>{tds}</tr>"
    return (f"<div style='max-height:380px;overflow:auto'>"
            f"<table style='border-collapse:collapse;font-size:13px;width:100%'>"
            f"<tr>{th}</tr>{body}</table></div>"
            f"<div style='color:#888;font-size:12px;margin-top:4px'>"
            f"{len(rows)} row(s)</div>")

# 2. GUI LAYER 

In [7]:
def build_ui():
    import ipywidgets as w

    # --- status line ---
    status = w.HTML(value="")

    def say(msg, ok=True):
        color = "#1a7f37" if ok else "#c62828"
        status.value = (f"<div style='padding:6px 10px;border-left:4px solid {color};"
                        f"color:{color}'>{html.escape(msg)}</div>")

    # --- insert panel ---
    wide = w.Layout(width="420px")
    tournament_in = w.Text(description="Tournament", placeholder="e.g. Tata Steel Masters 2026", layout=wide)
    email_in = w.Text(description="Arbiter Email", placeholder="only needed for a NEW tournament", layout=wide)
    player_in = w.Text(description="Player", placeholder="e.g. D. Gukesh", layout=wide)
    type_in = w.Combobox(
        description="Record Type", placeholder="pick or type a record type",
        options=["Opponent Rating", "Moves Played", "Points Scored"],
        ensure_option=False, layout=wide)
    value_in = w.Text(description="Record Value", placeholder="non-negative number", layout=wide)
    add_btn = w.Button(description="Add Record", button_style="success", icon="plus")

    # --- delete panel ---
    kind_dd = w.Dropdown(description="Delete a", options=list(DELETE_KINDS), layout=wide)
    target_dd = w.Dropdown(description="Item", options=[], layout=w.Layout(width="620px"))
    del_btn = w.Button(description="Delete Selected", button_style="danger", icon="trash")

    # --- tabs ---
    views = {name: w.HTML() for name in VIEW_QUERIES}
    tabs = w.Tab(children=list(views.values()))
    for i, name in enumerate(VIEW_QUERIES):
        tabs.set_title(i, name)

    # ---- refresh logic ----
    _busy = {"flag": False}  # stops the kind observer firing mid-refresh

    def refresh_delete_dropdown():
        _busy["flag"] = True
        try:
            opts = get_delete_options(kind_dd.value)
            if opts:
                target_dd.options = opts
                target_dd.value = opts[0][1]      # always a valid, existing id
                target_dd.disabled = False
                del_btn.disabled = False
            else:
                target_dd.options = [("- nothing to delete -", None)]
                target_dd.value = None
                target_dd.disabled = True
                del_btn.disabled = True
        finally:
            _busy["flag"] = False

    def refresh_views():
        for name, sql in VIEW_QUERIES.items():
            views[name].value = query_table_html(sql)

    def refresh_all():
        refresh_views()
        refresh_delete_dropdown()

    # ---- handlers ----
    def on_add(_):
        try:
            msg = add_record(tournament_in.value, player_in.value, type_in.value,
                             value_in.value, email_in.value)
        except (ValueError, sqlite3.Error) as e:
            say(str(e), ok=False)       # keep inputs so the user can fix them
            return
        tournament_in.value = email_in.value = player_in.value = ""
        type_in.value = value_in.value = ""
        say(msg)
        refresh_all()

    def on_delete(_):
        row_id = target_dd.value
        if row_id is None:
            say("Nothing selected to delete.", ok=False)
            return
        try:
            msg = delete_row(kind_dd.value, row_id)
        except (ValueError, sqlite3.Error) as e:
            say(str(e), ok=False)
            refresh_all()
            return
        say(msg)
        refresh_all()   # rebuilds dropdown from live data -> no stale IDs

    def on_kind_change(change):
        if not _busy["flag"]:
            refresh_delete_dropdown()

    add_btn.on_click(on_add)
    del_btn.on_click(on_delete)
    kind_dd.observe(on_kind_change, names="value")

    refresh_all()

    def card(title, *children):
        return w.VBox(
            [w.HTML(f"<b style='font-size:15px'>{title}</b>"), *children],
            layout=w.Layout(border="1px solid #cfd6e4", padding="10px 14px",
                            margin="0 0 12px 0"))

    return w.VBox([
        w.HTML("<h2 style='margin:0 0 8px 0'>Chess Tournament & Player Performance Dashboard</h2>"),
        card("Insert Panel", tournament_in, email_in, player_in, type_in, value_in, add_btn),
        card("Delete Panel", kind_dd, target_dd, del_btn),
        status,
        tabs,
    ])

# 3. INITIALISATION BLOCK

In [8]:
init_db()
seed_if_empty()

try:
    from IPython.display import display
    display(build_ui())
except ImportError:
    print("ipywidgets/IPython not found. Run `pip install ipywidgets` and "
          "execute this script in Jupyter, VS Code notebook cells, or Colab.")